# Baseline benchmark — what ran, where it landed, what it means

Three runs live in this notebook, in the order they were done. They answer the same question —
*how much of CellOT's accuracy needs CellOT?* — under different protocols, and **they are not
row-compatible with each other**; never merge their CSVs.

| part | run | protocol | dir | date |
|---|---|---|---|---|
| A, §1–§9 | stim arm, LOPO | baselines refit 19×, one patient out each time | `lopo/`, `lopo_test/` | 2026-09-16/17 |
| **B, §10** | **stim arm, pooled** | **CellOT's own 80/20 cell split, fit once, score once** | `pooled/` | 2026-09-18 |
| C, §11 | drug arm, pooled | DMSO → drug within a stim, 9 drugs × 168 pairs | `pooled_drug/` | 2026-09-20 |
| §12 | selection leak | `model.pt` (argmin checkpoint) vs `last.pt`, both arms | `selection_leak_*/` | 2026-09-19/21 |

**Part B is the one to quote.** It is CellOT's own protocol applied unchanged to the baselines,
so training cells and test cells are equalised between them; Part A's LOPO is stricter on the
baselines only (they are refit without a patient, CellOT never is), which flatters CellOT.
Part A also predates `gauss_ot` / `linear_median` / `linear_median_ridge` — it has three
baselines, Part B and C have six.

**Kernel: `CellOT (py3.9)`** (registered at `~/.local/share/jupyter/kernels/cellot`, pointing
at `/scratch/users/bmerwan/venv-cellot`). The analysis cells only need pandas/numpy/matplotlib,
but the `%%bash` cell in §9 shells out to `baselines.summarize`, which imports `cellot.losses` —
that one needs the venv on `PATH`, so start JupyterLab from it or use this kernel throughout.

Everything here is read-only and takes seconds except the `%%bash` in §9 (it regenerates summary
CSVs, idempotent). The genuinely expensive re-runs are left as markdown, not code, so "Run All"
cannot start a multi-hour job — the commands that *produced* every CSV read below are recorded
verbatim in §10.0, §11.0 and §13.

## 1. Where everything comes from

### "LOPO" applies to the baselines only — read this first

The two arms of the table are produced by **different protocols**, and the name is
misleading if taken to cover both:

| arm | protocol | out-of-sample in |
|---|---|---|
| `identity`, `median_shift`, `quantile_map` | **19 separate fits** per (stim, celltype), each excluding one patient (`run_baselines.py:218-221`) | cells **and** patients |
| `cellot_insample` | **one** checkpoint, loaded outside the patient loop (`:208-216`) and applied unchanged to every patient (`:229`) | cells only |

CellOT is **not cross-validated here, or anywhere in this tree.** It was trained once,
under a single `train_test` split, on cells from all 19 patients. The per-patient loop in
this benchmark is a *partition of the evaluation set* for CellOT, not a refit. Reporting
its error broken down by held-out patient does not make it a held-out estimate.

What CellOT's split actually is, verified in the training code:

- `datasplit: {groupby: stim, name: train_test, test_size: 0.2}`, `random_state` absent → 0
  (`cellot/data/cell.py:384-396`). Grouping key is `split_on = ["split", "transport"]`
  (`cell.py:322-326`) — **no patient dimension**.
- training draws from `iterator.train.*` — the 80% (`train.py:203-204`, used `:216`, `:235`)
- `evaluate()` draws from `iterator.test.*` — the 20% (`train.py:58-60`), every
  `eval_freq: 250` steps
- `model.pt` is overwritten whenever that test metric improves (`train.py:255-265`,
  `metric: median`)

So the 20% is the **model-selection set**: the shipped checkpoint is the argmin over ~200
evaluations of test-set `median_error`. Its error there is biased low before any patient
consideration.

### The two passes

They differ in **one flag**.

| Pass | Launcher | Flag | Per-celltype CSVs | Summary |
|---|---|---|---|---|
| leaky | `baselines/run_local.sh` | — | `simco_work/baselines/lopo/` | `summary_all/` |
| **no cell leak** | `baselines/run_local_testsplit.sh` | `--eval-split test` | `simco_work/baselines/lopo_test/` | `summary_test/` |

In the leaky pass everyone is scored on all of the held-out patient's cells, 80% of which
CellOT took gradient steps on. `--eval-split test` (`run_baselines.py:115-138`) re-derives
that deterministic split and scores everyone on the 20% only.

**It removes the training leak, not the selection leak** — those same cells are in the pool
CellOT's checkpoint was chosen on. Modest in size (each `evaluate()` is one 256-cell batch,
and the held-out patient is ~1/19 of the pool) but structural. See §8.

### Run status, Sep 16 2026

Both passes ran *inside a JupyterLab allocation*, not via `sbatch`, so `sacct`/`squeue` show
nothing for them. The no-leak pass finished 22:17; the leaky pass's last celltype
(Granulocytes) finished 23:19.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

REPO = Path("/scratch/users/bmerwan/simco")
WORK = Path("/scratch/users/bmerwan/simco_work/baselines")
DATA = Path("/home/groups/gbrice/ptb-drugscreen/ot/cellot/datasets")
PAPER_RUN = Path(
    "/home/groups/gbrice/ptb-drugscreen/ot/cellot/results/cross_validation_original_13_HVPV"
)

# The no-leak pass is the one to quote. Switch to "lopo"/"summary_all" for the leaky pass.
LOPO, SUMMARY = WORK / "lopo_test", WORK / "summary_test"

print(LOPO, SUMMARY, sep="\n")

## 2. Did the run finish cleanly?

Three checks, because there is no Slurm exit code to look at:
1. one CSV per celltype,
2. every worker log ends with its `[WRITE]` line,
3. nothing in stderr beyond the known harmless `anndata` warning (exactly 230 bytes).

In [ ]:
prefix = "test" if LOPO.name == "lopo_test" else "local"
expected = (WORK / "celltypes.txt").read_text().split()
present = sorted(p.stem for p in LOPO.glob("*.csv"))

print(f"{len(present)} CSVs in {LOPO.name}/  (celltypes.txt lists {len(expected)})")
missing = [c for c in expected if c not in present]
print("missing:", missing or "none")

wrote = [p for p in WORK.glob(f"logs/{prefix}_*.out")
         if p.read_text().rstrip().rsplit("\n", 1)[-1].startswith("[WRITE]")]
print(f"logs ending in [WRITE]: {len(wrote)}")

noisy = [(p.name, p.stat().st_size) for p in WORK.glob(f"logs/{prefix}_*.err")
         if p.stat().st_size != 230]
print("stderr beyond the 230-byte anndata warning:", noisy or "none")

**`Granulocytes` is expected to be missing from the no-leak pass.** Both launchers iterate
`celltypes_small.txt` (27 entries), which excludes it — its `.h5ad` is the ~30 GB outlier and it
was run separately, *leaky pass only*, via `bash baselines/run_local.sh Granulocytes`.

So the no-leak numbers below cover **161 of 168** `(stim, celltype)` pairs.

## 3. The raw table

`summary_test/all_rows.csv` is just the per-celltype CSVs concatenated. One row per
**`(celltype, stim, held-out patient, method)`**, so 4 rows per evaluation.

In [ ]:
df = pd.read_csv(SUMMARY / "all_rows.csv")

KEYS = ["celltype", "stim", "patient"]
METRICS = ["mean_error", "median_error", "w1", "mmd", "corr_error"]
CELLOT = "cellot_insample"
ORDER = ["identity", "median_shift", "quantile_map", CELLOT]

print(f"{len(df):,} rows")
print(f"{df.celltype.nunique()} celltypes x {df.stim.nunique()} stims "
      f"-> {df.groupby(['stim','celltype']).ngroups} (stim, celltype) pairs")
print(f"{df.patient.nunique()} patients -> "
      f"{df.groupby(KEYS).ngroups:,} paired evaluations, x4 methods = {len(df):,} rows")
df.head(4)[KEYS + ["method"] + METRICS]

### The four methods (`baselines/baselines.py`)

All three baselines are **diagonal** maps: they act on one marker at a time and cannot change the
marker–marker copula. Any CellOT advantage on a *joint* metric (`mmd`, `corr_error`) is therefore
attributable to cross-marker structure a diagonal map cannot represent.

| method | map | fit on |
|---|---|---|
| `identity` | `T(x) = x` — predict Unstim as the stim response | nothing |
| `median_shift` | `T(x) = x + d`, `d` one constant per marker | 18 training patients (`baselines.py:61`) |
| `quantile_map` | `T_j(x) = F_target^-1(F_source(x))`, per marker | 18 training patients (`baselines.py:81`) |
| `cellot_insample` | the paper's trained transport map | **all 19 patients** — including the held-out one |

The last row is the whole caveat: CellOT is *not* held out. It is an upper bound on CellOT's
held-out performance, not an estimate of it.

### The five metrics (`baselines/metrics.py`)

All lower = better. All computed on one `(method, stim, celltype, held-out patient)` cell set.

| metric | definition | notes |
|---|---|---|
| `mean_error` | `\|mean(pred) - mean(true)\|` per marker, averaged over the 13 markers | |
| `median_error` | `\|median(pred) - median(true)\|` per marker, averaged | **error *of* the median, not median of errors** — reproduces the repo's own training metric (`cellot/train/train.py`, `metric: median`) |
| `w1` | 1-D Wasserstein-1 per marker, averaged | the objective `quantile_map` is *built* to minimise, so any CellOT win here comes from elsewhere |
| `mmd` | `cellot.losses.compute_scalar_mmd`, RBF kernel | the only **joint** distributional metric; 2000-cell subsample, fixed seed, identical cells for every method |
| `corr_error` | Frobenius norm of the marker–marker correlation matrix difference | isolates cross-marker structure; identical for `identity` and `median_shift` by construction (a shift cannot change a correlation) |

The table also carries per-marker breakdowns: `median_error__<marker>`, `mean_error__<marker>`,
`w1__<marker>`. See §7 — that granularity is the one that matters for Stabl.

## 4. `all` vs `live` vs `null` — the row subset

This is a filter on `(stim, celltype)` pairs, nothing else. Same metrics, same methods, fewer rows.

`identity` predicts the held-out patient's **measured Unstim** cells as the stim response. So
`identity`'s MMD *is* the distance between that patient's unstimulated and stimulated cells —
i.e. **the size of the real stimulation effect**. Where it's ~0, the stim did nothing, there is no
response to predict, and every method is fitting noise.

`summarize.py:60` splits on that at `identity mmd >= 0.005`:

- **`all`** — every pair.
- **`live`** — pairs where the stimulation actually does something. *This is the honest subset.*
- **`null`** — the dead pairs, kept separately so they can be inspected rather than silently
  diluting the pooled average.

Pooling them hides what the methods do when there is something to learn: on a dead pair the
*correct* answer is "change nothing", which flatters whichever method moves least.

In [ ]:
def split_live_null(df, threshold=0.005):
    """Reproduces summarize.py:60-72."""
    effect = df[df.method == "identity"].groupby(["stim", "celltype"])["mmd"].mean()
    live_idx = effect[effect >= threshold].index
    key = pd.MultiIndex.from_frame(df[["stim", "celltype"]])
    return df[key.isin(live_idx)], df[~key.isin(live_idx)], effect


live, null, effect = split_live_null(df)
n_live = int((effect >= 0.005).sum())
print(f"live pairs: {n_live}/{len(effect)}   null pairs: {len(effect) - n_live}")
print(f"live evaluations: {live.groupby(KEYS).ngroups:,}   "
      f"null evaluations: {null.groupby(KEYS).ngroups:,}")

per_stim = effect.groupby(level=0).agg(
    pairs="size",
    live=lambda g: int((g >= 0.005).sum()),
    median_effect="median",
)
per_stim

The null pairs are not spread evenly — they are almost entirely **IL33** (8/27 live) and
**TNFa** (12/18). Those two stims barely move these cells at all in this dataset, which is worth
knowing before reading any per-stim result for them.

## 5. View 1 — raw means (`raw_means_*.csv`)

The average error each method makes, in the metric's own units.

**Caveat, from `summarize.py`'s own docstring:** not comparable across celltypes, because marker
scales differ. The average leans on whichever celltypes everyone does badly on.

In [ ]:
def raw_means(d):
    return d.groupby("method")[METRICS].mean().reindex(ORDER)


for label, subset in [("all", df), ("live", live), ("null", null)]:
    print(f"=== raw means, {label} ===")
    print(raw_means(subset).to_string(float_format=lambda v: f"{v:9.4f}"), "\n")

## 6. View 2 — paired win rates (`win_rate_vs_cellot_*.csv`)

**This is the second table.** It answers a different question from §5 and the two disagree, which
is the interesting part.

Pivot the rows to `(celltype, stim, patient) x method`, then for each baseline count the fraction
of those evaluations where **that baseline's error is smaller than CellOT's on the exact same
evaluation** (`summarize.py:43`). It is a *paired* comparison — same celltype, same stim, same
patient, same cells.

- `0.50` = coin flip.
- `0.00` = CellOT wins every single time.
- No normalisation, no ratios, nothing that can blow up on a near-zero denominator. That
  robustness is the reason it exists alongside the means.

**It measures how *often*, never by how much.** A method can sit at 0.50 here and still have a much
worse mean in §5, if CellOT's wins are large and its losses are narrow. That is exactly what
`median_shift` does.

In [ ]:
def win_rates(d):
    wide = d.pivot_table(index=KEYS, columns="method", values=METRICS)
    return pd.DataFrame({
        m: {k: float((wide[(k, m)] < wide[(k, CELLOT)]).mean()) for k in METRICS}
        for m in ORDER if m != CELLOT
    }).T


for label, subset in [("all", df), ("live", live), ("null", null)]:
    print(f"=== win rate vs CellOT, {label}  (>0.5 = baseline beats CellOT more often than not) ===")
    print(win_rates(subset).to_string(float_format=lambda v: f"{v:9.3f}"), "\n")

## 7. The view that matters for Stabl: `median_error`

Stabl never sees a cell. Its feature matrix
(`ina_13OG_final_long_allstims_filtered_no_Unstim.csv`, 177 x 852, columns
`<celltype>_<marker>_<stim>`) is built by `scripts/compute_median_predictions.py`:

```python
medians = df_melt.groupby(["patient", "cell_type", "marker"])["value"].median()   # :186
merged["median_diff"] = merged["median_stim"] - merged["median_baseline"]          # :307
```

So one feature = `median(CellOT-predicted stim cells) - median(measured Unstim cells)`.

The baseline term is **measured**, not predicted, so it is identical on the predicted and the true
side and cancels in the difference. Therefore:

> the error in a Stabl feature **is** `median_error`, exactly — up to the fact that `median_error`
> averages over the 13 markers whereas Stabl treats each marker as its own column.

So use the per-marker columns for a faithful picture.

In [ ]:
markers = sorted(c.split("__", 1)[1] for c in df.columns if c.startswith("median_error__"))
print(f"{len(markers)} markers: {markers}\n")

cols = [f"median_error__{m}" for m in markers]
per_marker = live.groupby("method")[cols].mean().reindex(ORDER)
per_marker.columns = markers
print("=== mean |median(pred) - median(true)| per marker, live pairs ===")
print(per_marker.T.to_string(float_format=lambda v: f"{v:8.4f}"))

In [ ]:
# Per-marker paired win rate of median_shift vs CellOT -- is the tie uniform, or
# driven by a handful of markers?
wide = live.pivot_table(index=KEYS, columns="method", values=cols)
rates = pd.DataFrame({
    m: {
        c.split("__", 1)[1]: float((wide[(c, m)] < wide[(c, CELLOT)]).mean())
        for c in cols
    }
    for m in ORDER if m != CELLOT
})
print("=== per-marker win rate vs CellOT on median_error, live pairs ===")
print(rates.sort_values("median_shift", ascending=False)
           .to_string(float_format=lambda v: f"{v:8.3f}"))

In [ ]:
# Where does CellOT actually earn its advantage? Per (stim, celltype) margin on median_error:
# positive = CellOT better than median_shift.
w = live.pivot_table(index=KEYS, columns="method", values="median_error")
margin = (w["median_shift"] - w[CELLOT]).groupby(level=["stim", "celltype"]).mean()

print(f"CellOT better on {int((margin > 0).sum())}/{len(margin)} live pairs\n")
print("--- CellOT's 8 biggest wins ---")
print(margin.nlargest(8).to_string(float_format=lambda v: f"{v:+8.4f}"))
print("\n--- CellOT's 8 biggest losses ---")
print(margin.nsmallest(8).to_string(float_format=lambda v: f"{v:+8.4f}"))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(margin, bins=40, color="#4878A8", edgecolor="white")
axes[0].axvline(0, color="#B03A2E", lw=1.5)
axes[0].set_xlabel("median_shift error  -  CellOT error")
axes[0].set_ylabel("(stim, celltype) pairs")
axes[0].set_title("median_error margin, live pairs\n(>0 = CellOT better)")

for m in ORDER:
    axes[1].scatter(w[CELLOT], w[m], s=6, alpha=0.3, label=m)
lim = (0, float(np.nanpercentile(w.values, 99)))
axes[1].plot(lim, lim, color="#555", lw=1, ls="--")
axes[1].set(xlim=lim, ylim=lim, xlabel="CellOT median_error", ylabel="method median_error")
axes[1].set_title("paired, per held-out evaluation")
axes[1].legend(markerscale=3, fontsize=8)

fig.tight_layout()

### Reading the three cells above

**The 13 markers are not equal.** On `median_error`, `median_shift` — a per-marker *constant* —
matches or beats CellOT on 6 of 13 markers (`CD25` 0.60, `pSTAT1` 0.60, `HLADR` 0.54, `pMK2`,
`pNFkB`, `pSTAT6`, `pERK`, `pS6` all ≥ 0.50). CellOT's per-marker advantage is confined to
`pCREB`, `pSTAT5`, `pSTAT3`, `pp38`, and even there it is narrow.

**`quantile_map` has two blow-ups.** `pSTAT6` (0.424 vs CellOT 0.132) and `pSTAT1` (0.277 vs
0.102) are where it loses the aggregate. *Hypothesis, not tested here:* these are the two
markers with the most bimodal / responder-fraction-driven distributions, and quantile_map
averages quantile functions across training patients, so a patient-to-patient shift in the
responding fraction moves the barycentre's median between the modes. Checking it means plotting
the per-patient marginals for those two markers — worth doing before repeating the claim.

**CellOT's aggregate win is not broad, it is concentrated.** Across live pairs it beats
`median_shift` on 93/134, but the margin histogram is centred near zero: its wins cluster in
myeloid × strong-stim pairs (`PI`/`LPS`/`IL246` on `ncMCs`, `mDCs`, effector T cells) and it
*loses* on `IL246 × Bcells`, `PI × pDCs`, `PI × ncMCs`. Wins where the response is large and
structured, losses where it is small.

**What this means for Stabl.** Every one of these 852 features is one cell of this table. Where
CellOT ties a constant shift, the corresponding feature carries no more information than a
`(celltype, marker, stim)`-specific offset applied to the patient's own Unstim median — i.e. the
patient-level signal in that feature comes from the *measured* baseline, not from the transport
map. Worth checking which features Stabl actually selects against the per-marker table above.

## 8. Caveats — read before quoting any of this

1. **CellOT is not held out.** The `cellot_insample` column comes from models trained on all 19
   patients. Every "held-out" patient was in its training set and in none of the baselines'.
   These numbers are an upper bound on CellOT, not an estimate.

2. **`--eval-split test` removes the training leak, not the model-selection leak.** CellOT's
   checkpoint was *chosen* on those same 20% test cells: `cellot/train/train.py:255-265` evaluates
   every `eval_freq: 250` steps and keeps whichever step minimises `median_error`. The test pool is
   split by `stim` only — there is no `pair_batch_on` in the config — so it is pooled across all 19
   patients, including the held-out one. `cellot_insample` here is still optimistic.

3. **The 20% split is on cells, not patients.** Validation cells share patient, tube and batch with
   training cells, so it measures convergence, not generalization, and structurally cannot detect
   patient-level overfitting. Consistent with `CLAUDE.md`: no patient-level held-out evaluation
   exists anywhere in this tree.

4. **The split is re-derived, not recovered.** `run_baselines.py:116` recomputes a documented
   deterministic split (`random_state=0`). No split was ever saved, so it cannot be checked
   against an artefact.

5. **`Granulocytes` is absent from the no-leak pass** — 161/168 pairs, see §2.

6. **`summary_all/` (leaky pass) is stale.** It was written at 22:03, before Granulocytes finished
   at 23:19, so those tables cover 27 celltypes. Re-run the summarize cell in §9 before using it.

## 9. Re-running things

**Cheap** (seconds) — re-pool the per-celltype CSVs. Fixes the stale `summary_all/`:

In [ ]:
%%bash
cd /scratch/users/bmerwan/simco
W=/scratch/users/bmerwan/simco_work/baselines
python -m baselines.summarize --indir $W/lopo      --outdir $W/summary_all
python -m baselines.summarize --indir $W/lopo_test --outdir $W/summary_test

**Expensive — do not run in the notebook.** These need a compute allocation (`sh_dev -c 8`
or `salloc -p normal -c 8 --mem=64G --time=8:00:00`) and take hours:

```bash
ml load python/3.9.0 && source /scratch/users/bmerwan/venv-cellot/bin/activate
cd /scratch/users/bmerwan/simco

NPROC=4 bash baselines/run_local_testsplit.sh   # no-leak pass, 27 celltypes, ~3 h
NPROC=4 bash baselines/run_local.sh             # leaky pass, 27 celltypes
bash baselines/run_local.sh Granulocytes        # the 30 GB outlier, alone, ~1 h
```

`run_local_testsplit.sh` skips celltypes whose CSV already exists, so it is safe to re-run;
`run_local.sh` is not guarded and will redo everything.

To add `Granulocytes` to the **no-leak** pass, note the launcher takes no positional arguments
(unlike `run_local.sh`) and reads `celltypes_small.txt`, which excludes it. Either append it to
that file, or call the module directly:

```bash
python -m baselines.run_baselines \
  --celltype Granulocytes \
  --data-dir     $DATA/ptb_concatenated_batchcorrected \
  --features     $DATA/ptb_concatenated_per_condition_celltype/13features.txt \
  --pairs-file   $W/pairs_168.txt \
  --outdir       $W/lopo_test \
  --eval-split   test \
  --cellot-results $PAPER_RUN
```

---

# Part B — the pooled 80/20 protocol (`--eval-mode pooled`)

## 10.0 What ran, and the exact commands that produced these CSVs

Part A scores CellOT on a protocol CellOT was never fit under, and refits the baselines 19 times
while CellOT is loaded once. `pooled` removes that asymmetry by adopting **CellOT's own split**:

> fit each baseline once on the **80% train side** of CellOT's cell split, pooled over all 19
> patients; transport the **20% test side**; score once.

One row per `(stim, celltype, method)` — the `patient` column is the literal string `"pooled"`.
The split is over *cells*, grouped by `stim`, with **no patient dimension**, so both halves hold
all 19 patients: **there is no held-out patient here and none is claimed.** What it does equalise
is training cells and test cells between the baselines and CellOT. The one thing it cannot
equalise is that CellOT's checkpoint is the argmin over ~200 evaluations *on these same test
cells* — quantified in §12.

Transport happens patient-by-patient then concatenates: a no-op for the row-wise maps, but
required by `linear_median*`, which are therefore the only methods whose `corr_error` moves in
pooled mode while being pinned to identity's value in LOPO.

```bash
# produced simco_work/baselines/pooled/  — 28 CSVs, one per celltype, ~2 h at NPROC=4
ml load python/3.9.0 && source /scratch/users/bmerwan/venv-cellot/bin/activate
cd /scratch/users/bmerwan/simco
NPROC=4 bash baselines/run_pooled.sh                  # 27 celltypes, smallest first
bash baselines/run_pooled.sh Granulocytes             # the 30 GB outlier, on its own

# produced the summary_pooled_* table dirs (seconds; re-runnable, and re-run below)
W=/scratch/users/bmerwan/simco_work/baselines
python -m baselines.summarize --indir $W/pooled --outdir $W/summary_pooled_cell       --weighting cell
python -m baselines.summarize --indir $W/pooled --outdir $W/summary_pooled_unweighted --weighting unweighted
```

`run_pooled.sh` skips a celltype whose CSV already exists, so it is safe to re-run; it pins
`--eval-mode pooled --eval-split test --weighting both` and points `--cellot-results` at
`results/cross_validation_original_13_HVPV` (the paper's run — the only `cross_validation_*` dir
carrying `preds_*`). `--eval-split test` is **enforced** in pooled mode.

The cells below recompute every published table from `pooled/*.csv` directly, so nothing here
depends on the summary dirs being current. They reproduce `summary_pooled_cell/` and
`summary_pooled_unweighted/` exactly (checked value-for-value, 2026-09-21).

In [ ]:
# Part B/C helpers. Deliberately named apart from the Part A functions above so that
# re-running cells out of order cannot cross-contaminate the two protocols.
POOLED, POOLED_DRUG = WORK / "pooled", WORK / "pooled_drug"

METRICS = ["mean_error", "median_error", "w1", "mmd", "corr_error"]
CELLOT = "cellot_insample"
# Roughly increasing capacity: fit-free, diagonal, marker-mixing, patient-adaptive, CellOT.
ORDER = ["identity", "median_shift", "quantile_map", "gauss_ot",
         "linear_median", "linear_median_ridge", CELLOT]


def load_dir(d):
    """Concatenate the per-celltype CSVs. Mirrors summarize.py:load()."""
    files = sorted(Path(d).glob("*.csv"))
    if not files:
        raise SystemExit(f"no result CSVs in {d}")
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


def pair_keys(df):
    """The unit an effect is measured on. The drug arm carries a `drug` column the stim
    arm lacks; without it the nine drugs of one (stim, celltype) silently collapse into
    one pivot cell and every win rate is taken against a mean over drugs."""
    return ["stim", "celltype"] + (["drug"] if "drug" in df.columns else [])


def keep_arm(df, weighting):
    """Keep one weighting arm. identity is fit-free and CellOT is fixed at (all, cell) by
    construction, so both are emitted once with weighting='invariant' and must survive."""
    return df[df.method.isin([CELLOT, "identity"]) | (df.weighting == weighting)]


def pair_split_live_null(df, threshold=0.005):
    """identity's MMD *is* the size of the real perturbation (untreated vs measured treated).
    Below the threshold there is no response to predict and every method is fitting noise."""
    keys = pair_keys(df)
    effect = df[df.method == "identity"].groupby(keys)["mmd"].mean()
    live = effect[effect >= threshold].index
    key = pd.MultiIndex.from_frame(df[keys])
    return df[key.isin(live)], df[~key.isin(live)], effect


def best_counts(d):
    """For each metric, how many pairs each method wins outright (argmin over methods)."""
    wide = d.pivot_table(index=pair_keys(d), columns="method", values=METRICS)
    cols = [c for c in ORDER if c in set(d.method)]
    out = {m: wide[m].reindex(columns=cols).idxmin(axis=1).value_counts() for m in METRICS}
    return pd.DataFrame(out).reindex(cols).fillna(0).astype(int)


def paired_win_rates(d):
    """Fraction of pairs where a baseline's error is smaller than CellOT's on that same
    pair. No normalisation, no ratios; 0.50 is a coin flip. Mirrors summarize.py:win_rates()."""
    wide = d.pivot_table(index=pair_keys(d), columns="method", values=METRICS)
    rows = {meth: {m: float((wide[(m, meth)] < wide[(m, CELLOT)]).mean()) for m in METRICS}
            for meth in ORDER if meth != CELLOT and meth in set(d.method)}
    return pd.DataFrame(rows).T


fmt = lambda v: f"{v:9.4f}"
pct = lambda v: f"{v:8.3f}"
print("helpers defined")

## 10.1 Did the run finish, and what is in it

In [ ]:
stim = load_dir(POOLED)
print(f"{len(sorted(POOLED.glob('*.csv')))} celltype CSVs, {len(stim):,} rows, "
      f"{stim.groupby(['stim', 'celltype']).ngroups} (stim, celltype) pairs, "
      f"{stim.method.nunique()} methods")
print("patient column :", stim.patient.unique())          # 'pooled' -- no held-out patient
print("eval_split     :", stim.eval_split.unique())       # 'test'  -- the 20% side
print("\narms present (fit_patients x weighting):")
print(stim[["method", "fit_patients", "weighting"]].drop_duplicates()
          .sort_values(["weighting", "method"]).to_string(index=False))

`identity` and `cellot_insample` are emitted once as `invariant`: identity is fit-free, and
CellOT's checkpoint is fixed at (all patients, cell-weighted) by construction. Everything else
appears twice, once per weighting arm — §10.6 is why that matters.

## 10.2 live vs null — 41 of 168 pairs have nothing to predict

`identity` predicts the measured Unstim cells as the stim response, so **its MMD is the size of
the real stimulation effect**. Pairs below 0.005 are near-null: no signal, every method fitting
noise. Pooling them with the rest hides what the methods do when there *is* something to learn.

In [ ]:
cell = keep_arm(stim, "cell")
live, null, effect = pair_split_live_null(cell)
n_live = int(effect.ge(0.005).sum())
print(f"{n_live} of {len(effect)} pairs live (identity MMD >= 0.005); "
      f"{len(effect) - n_live} near-null\n")

by_stim = pd.DataFrame({
    "pairs": effect.groupby("stim").size(),
    "live": effect.ge(0.005).groupby("stim").sum(),
    "median_identity_mmd": effect.groupby("stim").median(),
    "max_identity_mmd": effect.groupby("stim").max(),
})
print(by_stim.to_string(float_format=fmt))

**All 28 IL33 pairs are near-null** — an independent reproduction of the carve-out the paper's
own pair-selection rule implied (`CLAUDE.md`, "The 168-pair grid": IL33 was admitted at a
threshold of 0.01 where every other stim had to clear 0.1, and its largest effect anywhere is
0.083). Two runs, different evidence, same conclusion: **IL33 should not be read as a result.**
TNFa contributes most of the remaining 13.

## 10.3 The headline — each method wins the statistic it optimises

Two views of the same 127 live pairs. `best in N` is an outright argmin over the seven methods;
the win rate is **paired** against CellOT on the identical pair, so it is unaffected by a
method being mediocre everywhere else.

In [ ]:
print("=== best method, live pairs, cell-weighted (outright argmin over 7 methods) ===")
print(best_counts(live).to_string())
print("\n=== paired win rate vs CellOT, live pairs, cell-weighted ===")
print("    (>0.5 = that baseline beats CellOT on more than half the pairs)")
print(paired_win_rates(live).to_string(float_format=pct))

In [ ]:
print("=== mean error per method, live pairs, cell-weighted (metric's own units) ===")
print(live.groupby("method")[METRICS].mean().reindex(ORDER).to_string(float_format=fmt))
print("\nCaveat from summarize.py's docstring: raw means are NOT comparable across celltypes")
print("(marker scales differ), so the average leans on whichever celltypes everyone fails on.")

### Reading §10.3

**CellOT keeps the joint metrics and only those.** It wins `mmd` outright on 106/127 and `w1` on
68/127 — it is the only method fit to a joint objective, and that is where the advantage is real.

**It loses every marker-wise metric, and one loss is sharper than the others:**

- `median_shift` — a per-marker *constant* — beats it on `median_error` in **98.4%** of live
  pairs (0.0090 vs 0.0271). `median_error` is **CellOT's own checkpoint-selection metric**
  (`data.metric: median` in all 168 dumped configs; `cellot/train/train.py:258-263`). So a
  constant shift beats the trained model on the criterion that picked it, on the cells that
  picked it.
- `gauss_ot` — one 13×13 affine Bures–Wasserstein map, exactly optimal for Gaussians — beats it
  on `mean_error` in **96.9%** (0.0081 vs 0.0228) and on `corr_error` in **95.3%** (0.220 vs
  0.327). **Cross-marker structure is therefore not where CellOT's edge lives**: an affine map
  captures the marker–marker correlations better than the neural transport does.
- Patient-adaptivity does not pay: `linear_median` / `linear_median_ridge` never lead. Ridge α
  never pins at the top of its path (per-stim medians 0.02–0.22), so the patient-level signal is
  real but small.

**Why this matters downstream:** every one of Stabl's 852 features is
`median(CellOT-predicted stim) − median(measured Unstim)`, i.e. exactly the `median_error` cell of
this table (§7). Where a constant shift ties CellOT, the patient-level signal in that feature
comes from the *measured* baseline, not from the transport map.

## 10.4 Which (stim, celltype) pairs separate the methods

§10.3 counts pairs; this asks *which* pairs, and by how much. For every live pair it takes
CellOT's error and the best baseline's error on that same pair, and reports the margin

    margin = best_baseline_error − cellot_error        (> 0 = CellOT better)

Read `median_error` and `mmd` first: they are the two metrics that disagree, and the
disagreement is the whole result.

In [ ]:
def margins(d, metric):
    """Per-pair CellOT error, best baseline's error, which baseline it was, and the margin."""
    wide = d.pivot_table(index=pair_keys(d), columns="method", values=metric)
    base_cols = [c for c in ORDER if c != CELLOT and c in wide.columns]
    base = wide[base_cols]
    out = pd.DataFrame({
        "cellot": wide[CELLOT],
        "best_baseline": base.min(axis=1),
        "which": base.idxmin(axis=1),
    })
    out["margin"] = out.best_baseline - out.cellot
    out["identity_mmd"] = d[d.method == "identity"].set_index(pair_keys(d))["mmd"]
    return out.sort_values("margin")


for metric in ["median_error", "mmd"]:
    m = margins(live, metric)
    won = int((m.margin > 0).sum())
    print(f"########## {metric}: CellOT beats every baseline on {won}/{len(m)} live pairs "
          f"(median margin {m.margin.median():+.4f})")
    print("\n--- CellOT's 10 worst pairs (most negative margin = beaten by the widest gap) ---")
    print(m.head(10).to_string(float_format=fmt))
    print("\n--- CellOT's 10 best pairs ---")
    print(m.tail(10).iloc[::-1].to_string(float_format=fmt))
    print()

**The two rankings are largely the *same* pairs, ordered in opposite directions** — which is
the sharpest version of the §10.3 result. `IL246 × Bcells` is CellOT's single **worst**
`median_error` pair (margin −0.0577, beaten by `median_shift`) and its single **best** `mmd` pair
(+0.0073). `IL246 × CD8Tnaive` and `IL246 × CD4Teff` do the same thing. On exactly the pairs
where the stimulation restructures the joint distribution hardest (`identity_mmd` 0.09–0.18, the
top of the range), CellOT tracks that structure better than anything else here **and** places the
per-marker medians worse than a constant shift does.

Two more things the tables show:

- **CellOT beats every baseline on `median_error` in 1 of 127 live pairs** — `IL246 × pDCs`,
  by +0.0020. That is the whole of its marker-wise advantage under this protocol.
- **Its `mmd` losses are trivial in size.** The worst is −0.0010 and the rest are −0.0004 or
  smaller, almost all to `gauss_ot`; its `mmd` wins run an order of magnitude larger (+0.0073,
  +0.0062, +0.0039). Losing narrowly 21 times and winning widely 106 times is what the 106/127
  count in §10.3 is made of.

### The same thing as a grid — stim × celltype

One row per cell type, one column per stim, so a whole column going one way means the stim does
it and a whole row means the cell type does. Blank = near-null pair (dropped) or not in the 168.

In [ ]:
for metric in ["median_error", "mmd"]:
    grid = (margins(live, metric)["margin"]
            .unstack("stim")
            .reindex(columns=["IFNa", "IL246", "LPS", "PI", "TNFa", "GMCSF", "IL33"]))
    grid = grid.dropna(axis=1, how="all")
    print(f"=== margin on {metric} (best baseline − CellOT; >0 = CellOT better) ===")
    print(grid.to_string(float_format=lambda v: f"{v:+8.4f}", na_rep="     ."))
    print(f"  CellOT better in {int((grid > 0).sum().sum())} of {int(grid.notna().sum().sum())} "
          f"live pairs\n")

In [ ]:
# Which cell types and which stims separate the methods most, aggregated.
rows = []
for metric in ["median_error", "mmd"]:
    m = margins(live, metric)
    for level in ["celltype", "stim"]:
        g = m.groupby(level=level)["margin"]
        rows.append(pd.DataFrame({
            "metric": metric, "level": level,
            "n_live": g.size(), "mean_margin": g.mean(),
            "cellot_wins": m.assign(w=m.margin > 0).groupby(level=level)["w"].mean(),
        }))
agg = pd.concat(rows).set_index(["metric", "level"], append=True).reorder_levels([1, 2, 0])

for metric in ["median_error", "mmd"]:
    for level in ["stim", "celltype"]:
        sub = agg.loc[(metric, level)].sort_values("mean_margin")
        head = sub if level == "stim" else pd.concat([sub.head(5), sub.tail(5)])
        print(f"=== {metric} by {level} "
              f"({'all' if level == 'stim' else '5 worst / 5 best for CellOT'}) ===")
        print(head.to_string(float_format=fmt))
        print()

**Where the margins are widest.** On `median_error` it is `IL246`/`PI` on the lymphoid
populations — `Bcells`, `CD4Tregs`, `CD4Teff`, `CD8Tnaive` (mean margins −0.024 to −0.036) — not
Granulocytes. Granulocytes is a different kind of extreme and worth separating: its *margin* is
mid-pack (≈ −0.019), but the baseline's **absolute** error there is the smallest anywhere
(`median_shift` 0.0006 vs CellOT 0.0184, a 30× ratio), and it is also the largest cell type by
far — which is exactly why the cell-weighted rates in §10.6 are so lopsided.

On `mmd` the ordering is by effect size, not lineage: `IL246` has both the largest mean margin
(+0.0019) and the largest stimulation effects, and CellOT's per-cell-type `mmd` win rate hits
100% on `CD4Tem`, `mDCs` and `NK_cells_CD11cneg`. `TNFa` is the one stim where `mmd` is a coin
flip (0.50 over 8 live pairs), consistent with it being the other stim that barely moves these
cells.

A caution before reading any single row: these are **single pairs under one protocol with no
held-out patient**, so treat an individual margin as a pointer to look at that pair's marginals,
not as an effect size. The aggregate statements in §10.3 are what 127 pairs support.

## 10.5 Head-to-head: CellOT vs `median_shift` and `gauss_ot`, per stim and per cell type

§10.4 compares CellOT against *whichever* baseline won each pair. This section fixes the
opponent instead, because two baselines account for nearly all of the interesting behaviour:

- **`median_shift`** — one constant per marker. The cheapest thing that could work, and the one
  that beats CellOT on `median_error`, CellOT's own selection metric.
- **`gauss_ot`** — one 13×13 affine map. The cheapest thing that can move a marker–marker
  correlation, and the one that beats CellOT on `mean_error` and `corr_error`.

Values are the **metric's own units**, averaged over the live pairs in each group, so they are
comparable down a column but not across cell types (marker scales differ). `gap_*` is
`baseline − cellot`, so **positive = CellOT better**; `win_*` is the fraction of pairs in that
group where the baseline beat CellOT.

In [ ]:
CHALLENGERS = ["median_shift", "gauss_ot"]


def head_to_head(d, metric, level, methods=CHALLENGERS):
    """Mean metric value for CellOT and each named baseline, grouped by stim or celltype."""
    wide = d.pivot_table(index=pair_keys(d), columns="method", values=metric)
    t = pd.DataFrame({"cellot": wide[CELLOT], **{m: wide[m] for m in methods}})
    g = t.groupby(level=level).mean()
    for m in methods:
        g[f"gap_{m}"] = g[m] - g["cellot"]
        g[f"win_{m}"] = (t[m] < t["cellot"]).groupby(level=level).mean()
    g.insert(0, "n_live", t.groupby(level=level).size())
    return g


for metric in ["median_error", "mean_error", "corr_error", "w1", "mmd"]:
    print(f"########## {metric} by stim (live pairs, cell-weighted) ##########")
    print(head_to_head(live, metric, "stim")
          .sort_values("gap_median_shift").to_string(float_format=fmt), "\n")

In [ ]:
for metric in ["median_error", "mean_error", "corr_error", "mmd"]:
    t = head_to_head(live, metric, "celltype").sort_values("gap_median_shift")
    print(f"########## {metric} by celltype (all 28, sorted: CellOT worst vs median_shift "
          f"at the top) ##########")
    print(t.to_string(float_format=fmt), "\n")

In [ ]:
# Pair-level: the largest head-to-head differences, per challenger, on the metric it owns.
def top_gaps(d, metric, method, n=12):
    wide = d.pivot_table(index=pair_keys(d), columns="method", values=metric)
    t = pd.DataFrame({"cellot": wide[CELLOT], method: wide[method]})
    t["gap"] = t[method] - t["cellot"]
    t["ratio_cellot_over_base"] = t["cellot"] / t[method]
    return t.sort_values("gap")


for metric, method in [("median_error", "median_shift"),
                       ("mean_error", "gauss_ot"),
                       ("corr_error", "gauss_ot"),
                       ("mmd", "gauss_ot")]:
    t = top_gaps(live, metric, method)
    print(f"########## {metric}: CellOT vs {method} "
          f"({int((t.gap > 0).sum())}/{len(t)} pairs CellOT better) ##########")
    print(f"--- {method} ahead by the most ---")
    print(t.head(12).to_string(float_format=fmt))
    print(f"--- CellOT ahead by the most ---")
    print(t.tail(6).iloc[::-1].to_string(float_format=fmt), "\n")

### Reading §10.5

This is the table to quote a per-cell-type or per-stim number from — it names the opponent, so
there is no ambiguity about what "the baseline" was.

What to look for in it: whether CellOT's deficit on `median_error` is **uniform or concentrated**
(it is close to uniform — `win_median_shift` is 1.00 for almost every stim and cell type, which
is why the aggregate is 98.4%), and whether its `mmd` advantage survives per group (it does,
except on `TNFa`, the weakest live stim). Where `gap_gauss_ot` is positive on `mmd` while
`gap_median_shift` is positive too, CellOT is genuinely ahead of both on the joint objective in
that group — that is its real territory.

The `ratio_cellot_over_base` column in the pair-level tables is there because these errors span
two orders of magnitude across cell types: a gap of 0.02 means something different on
Granulocytes (where `median_shift` sits at 0.0006) than on `pDCs`.

## 10.6 The weighting qualifier — quote it with the numbers

`cell` weights each pair by its cell count (matching CellOT's cell-level sampling, so it is the
right arm for a parity claim); `unweighted` gives every pair equal say. The direction of every
result is the same, the magnitude is not.

In [ ]:
uw_live, _, _ = pair_split_live_null(keep_arm(stim, "unweighted"))
w_cell, w_uw = paired_win_rates(live), paired_win_rates(uw_live)

side = pd.concat({"cell": w_cell, "unweighted": w_uw}, axis=1)
side = side.reorder_levels([1, 0], axis=1).sort_index(axis=1)
print("=== paired win rate vs CellOT, live pairs, both weighting arms ===")
print(side.to_string(float_format=pct))

print("\nThe two comparisons quoted in CLAUDE.md:")
print(f"  median_shift beats CellOT on median_error : "
      f"cell {w_cell.loc['median_shift', 'median_error']:.1%}  "
      f"unweighted {w_uw.loc['median_shift', 'median_error']:.1%}")
print(f"  gauss_ot     beats CellOT on mean_error   : "
      f"cell {w_cell.loc['gauss_ot', 'mean_error']:.1%}  "
      f"unweighted {w_uw.loc['gauss_ot', 'mean_error']:.1%}")

Unweighted, the two headline rates fall to ~61% and ~58% — still above chance, far less dramatic.
The cell-weighted figures rest on the large celltypes (Granulocytes is the extreme case:
`median_shift` `median_error` 0.0006 vs CellOT 0.0184). **Never quote 98.4% / 96.9% without
saying "cell-weighted".**

## 10.7 The noise floor — how far apart are two samples that should be identical?

Every number in this notebook is a distance between a predicted and a measured population,
and none of them so far says how large a distance is **unavoidable**. This section measures that.

The data carries **three plate controls per (patient, stim)** — `DMSO1`, `DMSO2`, `DMSO3` — the
same cells under the same condition, differing only by handling. Scoring one against another
*with the identity map* is exactly the benchmark's `identity` row with the biology removed, so
what comes out is pure noise:

| `pair_type` | what it is | what it contains |
|---|---|---|
| `across_controls` | `DMSO_i` vs `DMSO_j`, same stim | plate/handling effects **+** finite-cell sampling |
| `split_half` | one control split in two at random | finite-cell sampling **alone**, at the same counts |

and two `scope`s: `pooled` (controls pooled over all 19 patients — what `--eval-mode pooled`
scores, so the directly comparable null) and `per_patient` (one donor at a time, which is the
form the question was asked in).

```bash
# produced simco_work/baselines/noise_floor/ -- 28 CSVs, ~1 min per small celltype
NPROC=3 bash baselines/run_noise_floor.sh
bash baselines/run_noise_floor.sh Granulocytes
```

No batch correction is applied: the drug arm's runtime correction shifts each patient's
per-marker medians onto a shared value, which would erase part of the variation being measured.

**On comparing these to the benchmark's numbers.** `mmd` is subsampled to 2000 cells a side by
`metrics.score` in *both* places, so where both sides have ≥2000 cells the MMD comparison is
like-for-like. The marker-wise metrics (`median_error`, `mean_error`, `w1`) use **all** cells and
are biased upward at small n, so compare them only with the `n_src_cells` column in view — a
floor computed on fewer cells than the benchmark's evaluation is an **over**estimate, i.e.
conservative for a "this is above the noise" claim.

In [ ]:
NOISE = WORK / "noise_floor"
nf = load_dir(NOISE)
print(f"{len(sorted(NOISE.glob('*.csv')))} celltype CSVs, {len(nf):,} rows, "
      f"{nf.celltype.nunique()} celltypes, controls {sorted(nf.source.unique())[:3]}...")
print("scopes:", dict(nf.scope.value_counts()), "| pair types:", dict(nf.pair_type.value_counts()))

nf_pooled = nf[nf.scope == "pooled"]
print("\n=== POOLED floor, median over celltypes, by stim and pair type ===")
print(nf_pooled.groupby(["stim", "pair_type"])[
          ["n_src_cells", "mmd", "median_error", "mean_error", "w1", "corr_error"]]
      .median().to_string(float_format=fmt))

### The floor is flat everywhere except under `PI` — and it is one control

`across_controls` should be ~`split_half` if the plates agree. It is, for every stim but one.

In [ ]:
ac = nf_pooled[nf_pooled.pair_type == "across_controls"]
print("=== pooled MMD between control pairs, median over celltypes ===")
print(ac.pivot_table(index="stim", columns=["source", "target"], values="mmd", aggfunc="median")
        .to_string(float_format=lambda v: f"{v:8.4f}"))

print("\n=== PI only, per celltype ===")
print(ac[ac.stim == "PI"].pivot_table(index="celltype", columns=["source", "target"], values="mmd")
        .to_string(float_format=lambda v: f"{v:8.4f}"))

**`DMSO2` and `DMSO3` agree under `PI` exactly as they do under every other stim (~0.003).
`DMSO1` does not** — it sits 0.05–0.09 away from both, in every cell type. Under every other
stim all three controls agree at ~0.002.

It is **not** that `DMSO1`'s PI wells failed. Restricting the stim-arm effect to one control
subset (one-off probe, `Unstim → PI`, `compute_scalar_mmd` on 2000-cell subsamples):

```text
celltype    controls       identity_mmd        ...and for LPS, as a control comparison
pDCs        all(1+2+3)           0.1063        0.0872
            2+3 only             0.1004        0.0936
            DMSO1 only           0.1758        0.0802
CD4Teff     all(1+2+3)           0.1531        0.0712
            2+3 only             0.1190        0.0651
            DMSO1 only           0.1979        0.0800
ncMCs       all(1+2+3)           0.0695        0.0742
            2+3 only             0.0648        0.0774
            DMSO1 only           0.1456        0.0806
```

So **the PI effect is real and large under every subset** — pooling the controls does not
manufacture it, and dropping `DMSO1` barely moves it. What differs is the *magnitude* of the PI
response: it is systematically **stronger** on `DMSO1`. LPS, run on the same plates, shows no
such split. So this is PI-specific plate-to-plate variation in response size, not a broken well.

**Why this matters for Part C.** Each drug is compared against its own plate control
(`use_DMSO` in `batchcorrection_2.csv`), so a drug effect is a within-plate comparison and is not
invalidated by the above. But look at which control each drug got:

In [ ]:
# Part C's frame is loaded here too, so this section runs standalone in top-to-bottom
# order (§11 reloads it under its own names; both are a ~1 s read of stored CSVs).
dr = load_dir(POOLED_DRUG)
dr_effect = pair_split_live_null(keep_arm(dr, "cell"))[2]

bc = (pd.read_csv(REPO / "cross_validation/batchcorrection_2.csv",
                  usecols=["Treatment", "use_DMSO"])
        .drop_duplicates().set_index("Treatment")["use_DMSO"])
live_by_drug = dr_effect[dr_effect >= 0.005].index.get_level_values("drug").value_counts()

tbl = pd.DataFrame({
    "control": bc.reindex(sorted(dr.drug.unique())),
    "live_pairs": live_by_drug.reindex(sorted(dr.drug.unique())).fillna(0).astype(int),
    "median_identity_mmd": dr_effect.groupby("drug").median(),
    "max_identity_mmd": dr_effect.groupby("drug").max(),
}).sort_values("live_pairs", ascending=False)
print("=== drug arm: live pairs vs which plate control the drug was compared against ===")
print(tbl.to_string(float_format=fmt))
print("\nlive pairs by control:",
      dict(tbl.groupby("control")["live_pairs"].sum()),
      "| PI share of live pairs:",
      f"{(dr_effect[dr_effect >= 0.005].index.get_level_values('stim') == 'PI').mean():.0%}")

**The two drugs that account for 39 of the 48 live pairs — `LPZ` and `SA` — are exactly the two
compared against `DMSO1`**, and 46 of the 48 live pairs are the `PI` stim: the one stim where
`DMSO1` behaves differently from the other two plates. The seven drugs on `DMSO2`/`DMSO3` share
9 live pairs between them, and `RIF` has none.

**The control swap was run, and it resolves this against the drug arm.**
`baselines/control_swap.py` rescores the `DMSO1` drugs against the other two controls under PI,
on the four cell types carrying 17 of the 48 live pairs (`simco_work/baselines/control_swap_PI.csv`).
Identity MMD, drug vs each control:

```text
cells       drug (own control)     vs DMSO1        vs DMSO2        vs DMSO3
raw         LPZ (DMSO1)          0.050-0.074     0.0008-0.0015   0.0027-0.0050
raw         SA  (DMSO1)          0.043-0.069     0.0016-0.0019   0.0050-0.0106
raw         CHT (DMSO2)          0.059-0.087     0.0024-0.0043   0.0012-0.0035
corrected   LPZ (DMSO1)          0.0058-0.0090   0.0016-0.0025   0.0050-0.0098
corrected   CHT (DMSO2)          0.0092-0.0244   0.0011-0.0015   0.0016-0.0037
```

- **`DMSO1` is the outlier against everything, drugs included.** `CHT` is referenced to `DMSO2`
  and has nothing to do with `DMSO1`, yet sits 0.059–0.087 from it. So this is not "plate 1
  responded more strongly and its drugs came along" — the `DMSO1` *control wells* are apart from
  every other population measured under PI.
- **`LPZ` is indistinguishable from `DMSO2`** (0.0008–0.0015 raw) while showing an apparent
  0.050–0.074 "effect" against its own control.
- **After correction it still decides live/null.** `LPZ` v `DMSO1` is 0.0058–0.0090, *above* the
  0.005 threshold; `LPZ` v `DMSO2` is 0.0016–0.0025, below it. Conversely `CHT` is not live
  against its own control (0.0011–0.0015) but would score 0.009–0.024 against `DMSO1`.

**So under PI a drug's live/near-null status is set by which control it was assigned, not by the
drug** — which is the origin of the pattern above. Treat the `LPZ`/`SA` PI results as
**unestablished** pending reanalysis against a clean control.

Also note what this says about *training*, not just scoring: the drug configs carry
`source: DMSO1, target: LPZ, stim: PI`, so that model's objective was largely "learn to undo the
`DMSO1` anomaly", and the learned map is read as the drug's effect. The same pairing is what the
baselines are fit to, so this is not a CellOT problem — it is a problem with what the pair is.

Limits: 4 cell types, PI only, pooled over patients; and if plate 1 differs *biologically* rather
than technically, a cross-plate comparison is not strictly valid either. But the symmetry — every
drug from every plate agreeing with `DMSO2`/`DMSO3` and disagreeing with `DMSO1` — points at the
`DMSO1` control wells specifically. Extending the swap to all 28 cell types needs no retraining.

### Floor vs signal — is the 0.005 live/null threshold in the right place?


In [ ]:
floor = (nf_pooled.groupby(["stim", "celltype", "pair_type"])["mmd"].median()
         .unstack("pair_type"))
# DMSO2-vs-DMSO3 on its own: the technical floor with the DMSO1/PI outlier taken out, so
# that PI's signal-to-noise is not judged against its own anomaly.
ac_all = nf_pooled[nf_pooled.pair_type == "across_controls"]
floor["controls_2v3"] = (ac_all[(ac_all.source == "DMSO2") & (ac_all.target == "DMSO3")]
                         .set_index(["stim", "celltype"])["mmd"])

cmp = pd.DataFrame({"identity_mmd": effect}).join(floor, how="left")
cmp["signal_over_technical"] = cmp.identity_mmd / cmp.across_controls
cmp["signal_over_2v3"] = cmp.identity_mmd / cmp.controls_2v3
cmp["signal_over_sampling"] = cmp.identity_mmd / cmp.split_half
cmp["live"] = cmp.identity_mmd >= 0.005

print("=== stim arm: measured effect vs the floors, median by stim ===")
print("  across_controls = all three control pairs; controls_2v3 = the clean pair only")
print(cmp.groupby("stim")[["identity_mmd", "across_controls", "controls_2v3", "split_half",
                           "signal_over_technical", "signal_over_2v3", "signal_over_sampling"]]
         .median().to_string(float_format=fmt))

print("\n=== how the 0.005 threshold sits against the floors (all 168 pairs) ===")
print(f"  median sampling floor (split_half)   : {cmp.split_half.median():.5f}"
      f"   -> threshold is {0.005 / cmp.split_half.median():.1f}x it")
print(f"  median technical floor (DMSO2 v DMSO3): {cmp.controls_2v3.median():.5f}"
      f"   -> threshold is {0.005 / cmp.controls_2v3.median():.1f}x it")
print(f"  pairs called live whose effect is below their own 2v3 floor: "
      f"{int((cmp.live & (cmp.identity_mmd < cmp.controls_2v3)).sum())}")
print(f"  pairs called null whose effect is above their own 2v3 floor: "
      f"{int((~cmp.live & (cmp.identity_mmd > cmp.controls_2v3)).sum())}")

print("\n=== drug arm against the same floors ===")
dcmp = pd.DataFrame({"identity_mmd": dr_effect}).join(floor, how="left")
print(f"  median drug effect                        : {dcmp.identity_mmd.median():.5f}")
print(f"  median sampling floor at those pairs      : {dcmp.split_half.median():.5f}")
print(f"  median technical floor (2v3) at those pairs: {dcmp.controls_2v3.median():.5f}")
print(f"  drug pairs below the sampling floor  : {(dcmp.identity_mmd < dcmp.split_half).mean():.1%}")
print(f"  drug pairs below the 2v3 floor       : {(dcmp.identity_mmd < dcmp.controls_2v3).mean():.1%}")


### Reading §10.7

Two floors, both from the `pooled` scope, medians over all pairs:

| floor | value (MMD) | what it is |
|---|---|---|
| sampling (`split_half`) | **0.00061** | two halves of one control well |
| technical (`DMSO2` v `DMSO3`) | **0.00130** | two different control wells, the clean pair |

- **The 0.005 live/null threshold is in a defensible place** — 8.2× the sampling floor and 3.8×
  the technical floor. **No** pair called live has an effect below its own technical floor. Sixteen
  pairs called *null* sit above theirs, so the threshold is conservative rather than generous:
  it discards a little real signal (mostly `TNFa`) rather than admitting noise.
- **IL33 is not a small effect, it is noise.** Its median effect is **0.70×** its own technical
  floor — two nominally identical control wells differ *more* than IL33 stimulation does. Third
  independent line to the same conclusion, after the paper's 10× looser selection threshold for
  IL33 and all 28 IL33 pairs landing near-null in §10.2.
- **`TNFa` is marginal, not null** — 3.2× the technical floor. It is the source of most of those
  16 conservative exclusions.
- **`PI` is a strong, real effect: 21× its clean floor.** Judged against the contaminated
  `across_controls` column it looks like 2.5×, which is an artefact of `DMSO1` being in that
  average — this is exactly why the two columns are reported separately.
- **The drug arm is mostly below the noise floor.** Median drug effect 0.00081 against a 0.00130
  technical floor: **72.2% of the 1512 drug pairs move the cells less than two identical control
  wells differ**, and 11.9% are below even the sampling floor. This is the quantitative form of
  §11.1's "the drugs barely move these cells" — and it is why no aggregate over all 1512 pairs
  means anything.
- **CellOT's marker-median error sits *at* the technical floor.** Between `DMSO2` and `DMSO3`
  (n ≈ 20.8k cells a side) `median_error` is **0.0273** and `mean_error` **0.0270**. On live
  pairs CellOT's are **0.0271** and 0.0228 — i.e. **CellOT's error in predicting a marker median
  is the same size as the difference between two nominally identical control wells.**
  `median_shift`'s 0.0090 is *below* that technical floor and within ~1.4× of the sampling floor
  (0.0067 at n ≈ 11k). Since each Stabl feature is exactly one of these medians (§7), that is
  the scale of the noise those 852 features carry.

  **Read with the n caveat**: the floors are computed at 11k/21k cells a side against the
  benchmark's ~59k, and all three metrics are n-dependent. The *sampling* floor would shrink
  roughly as 1/√n (to ~0.003 at 59k); the *technical* floor is a systematic plate offset and
  would not. A matched-n rerun is the clean way to state this and has not been done.


### Does the drug arm's batch correction already absorb this floor?

Partly — and the part it leaves behind matters. Two things have to be separated:

**The stim arm (§10.2–§10.6) applies no runtime correction at all.** `batch_correction` is
absent from all 168 dumped configs of `cross_validation_original_13_HVPV` (the `data:` block is
`condition/drug_used/features/metric/path/source/target/type` and nothing else). The floor above
is measured on exactly the cells those models were trained and scored on, so Part B's comparison
is matched.

**The drug arm does correct, at load time** (`cellot/data/cell.py:147-194`, replayed in
`baselines/drug_data.py:batch_correct`): each `(patient, population, treatment)` group's
per-marker median is shifted onto its own `correctedMedian` from `batchcorrection_2.csv`. Note
the controls are **not** forced together — `DMSO1`, `DMSO2` and `DMSO3` have *different*
`correctedMedian` values for the same `(patient, stim, marker, population)` (median spread 0.059,
90th pct 0.20), so the table preserves control-to-control differences by construction.

Applying that same correction to a control-vs-control pair (one-off probe):

```text
celltype  stim  DMSO1 vs DMSO2      mmd     median_error   mean_error
pDCs      PI    raw              0.0729         0.4397       0.4176
pDCs      PI    batch-corrected  0.0097         0.1055       0.0907
pDCs      LPS   raw              0.0026         0.0703       0.0727
pDCs      LPS   batch-corrected  0.0019         0.0557       0.0595
ncMCs     PI    raw              0.0489         0.4167       0.3966
ncMCs     PI    batch-corrected  0.0073         0.0928       0.1182
ncMCs     LPS   raw              0.0034         0.0655       0.0689
ncMCs     LPS   batch-corrected  0.0031         0.0628       0.0640
```

- **Under `PI` the correction removes ~85% of the anomaly** (0.073 → 0.010, 0.049 → 0.007) —
  it is doing real work on exactly the artefact §10.7 found.
- **It does not remove it.** The residual ~0.007–0.010 is still **5–8× the uncorrected technical
  floor elsewhere** (0.0013), and — the point that matters — it is **above the 0.005 live
  threshold** used on the drug arm, and above the **median live drug effect of 0.00716**. So
  roughly half of the 48 "live" drug pairs have an effect no larger than the residual difference
  between two control wells of the same condition, after correction.
- **Under ordinary stims the correction barely moves the floor** (0.0026 → 0.0019): it is a
  median shift, and most of this floor is distributional/sampling, which a median shift cannot
  touch.

So the answer to "isn't that already taken into account?" is: for the stim arm, no correction is
applied and nothing is taken into account; for the drug arm, the median offsets are, the rest is
not, and what remains is the same size as the effects being reported. **Caveat on §10.7's drug
lines:** the 72.2%/11.9% figures compare corrected drug effects against an *uncorrected* floor.
The matched version is the run to do before quoting them — rerun `noise_floor.py` with
`batch_correct` applied. On the evidence above it would move the non-PI floor very little and the
PI floor a great deal, in the direction that makes the PI drug pairs look *worse*, not better.

### What the correction actually does — lookup and shift, not interpolation

Worth stating precisely, because an intuitive reading ("drug A sits between `DMSO1` and `DMSO2`,
so shift it by a fraction of their difference") is **not** what runs here. `cellot/data/cell.py:147-194`,
replayed in `baselines/drug_data.py:batch_correct`, does, for each
`(patient, population, treatment, stim)` group and each marker:

```python
corrected = row["correctedMedian"]        # looked up per (Patient, Treatment, Stim, Marker, Population)
real      = np.median(data[mask, marker].X)
shift     = corrected - real
data[mask, marker].X += shift             # every cell of that group moves by the same amount
```

No well positions, no plate geometry, no weighting between controls. **Each treatment — every drug
*and* every DMSO — is moved onto its own target value independently.** The controls are not
interpolated between; they are corrected like everything else, which is why they still differ
afterwards (their `correctedMedian`s differ, median spread 0.059).

**Any interpolation logic lives upstream, in how `correctedMedian` was computed — and that code is
not in this repo.** `CLAUDE.md` records a "spline DMSO correction" behind Figures 2–3 whose source
is absent; a spline over plate/acquisition order fitted through the DMSO controls and evaluated at
each drug's position would produce exactly this table. That is a plausible reading of the
mechanism described, but **it cannot be verified here** — only the table's values survive.

**The consequence, stated carefully.** After the shift, a group's per-marker median *is* its
table value (the original code asserts `np.isclose` and warns otherwise). So for the **19
drug-screen patients**, the measured per-marker median effect of a drug equals
`correctedMedian(drug) − correctedMedian(its control)` — a function of `batchcorrection_2.csv`,
not of the cells. The cells still determine the shape of the distribution around that median,
which is what `mmd`, `w1` and CellOT's transport see.

**This is a statement about the *training targets*, not about the Stabl features** — an earlier
version of this note conflated the two. `batchcorrection_2.csv` covers exactly `HV04`–`HV11` and
`PV01`–`PV11`, the 19 patients of the drug screen. The table Stabl consumes
(`ina_13OG_final_long_allstims_filtered_no_Unstim.csv`, 177 rows) holds *different individuals*
(`99_G1`, `74_G2`, …, the OOL cohort), who have **no rows in the correction table and no measured
stim or drug response at all**. Their features are
`median(CellOT-predicted cells) − median(their own measured Unstim)`, produced by the model. So
the table fixes what the model was fit *to*; it does not fix what the model emits for anyone
else.

Measured on pDCs, per-marker median drug effect, all 19 patients × 13 markers, raw vs
post-correction (one-off probe):

```text
stim  drug  control    n   mean|raw|  mean|corrected|  shrink   corr(raw, corrected)
PI    LPZ   DMSO1    208     0.432        0.059         7.3x         0.08
PI    SA    DMSO1    208     0.395        0.032        12.4x        -0.05
PI    CHT   DMSO2    221     0.082        0.027         3.0x         0.51
PI    MF    DMSO2    221     0.076        0.014         5.3x         0.26
PI    RIF   DMSO3    195     0.069        0.020         3.5x         0.28
LPS   LPZ   DMSO1    234     0.111        0.050         2.2x         0.63
LPS   CHT   DMSO2    234     0.092        0.018         5.2x         0.34
IFNa  LPZ   DMSO1    234     0.088        0.042         2.1x         0.66
IFNa  MF    DMSO2    208     0.074        0.007        10.5x         0.06
```

Three things fall out:

1. **The correction shrinks median-level drug effects everywhere**, by 2–12×, not only on the
   anomalous plate.
2. **The raw `PI` effects of the two `DMSO1` drugs are ~5× everyone else's** (0.43 / 0.40 against
   0.07–0.08) — that is the `DMSO1` anomaly inflating the *uncorrected* comparison, and the
   correction is what removes it. After correction they are 0.059 / 0.032 against 0.014–0.027:
   still the largest, but the gap narrows from ~5× to ~2×.
3. **For exactly those two pairs the corrected effect is uncorrelated with the raw one**
   (corr 0.08 and −0.05, against 0.26–0.66 elsewhere). Where the correction has the most work to
   do, what comes out is determined by the table rather than by the measured cells.

### The floor per marker — and does it have a direction?

`metrics.score` writes `median_error__<marker>`, `mean_error__<marker>` and `w1__<marker>` into
every row, so the floor breaks down per marker for those three. **`mmd` does not** — it is a
single joint distance over all 13 markers at once, so there is no `mmd__<marker>`.

Read the `zero_exactly` column first: for zero-inflated markers both sides can have a median of
*exactly* 0, making `median_error` exactly 0 and the median-over-pairs misleadingly clean. That
is why the mean is shown alongside.

In [ ]:
sh_p = nf_pooled[nf_pooled.pair_type == "split_half"]
ac_p = nf_pooled[nf_pooled.pair_type == "across_controls"]
c23 = ac_p[(ac_p.source == "DMSO2") & (ac_p.target == "DMSO3")]
c12 = ac_p[(ac_p.source == "DMSO1") & (ac_p.target == "DMSO2")]
MARKERS = [c.split("__", 1)[1] for c in nf.columns if c.startswith("median_error__")]

rows = {}
for mk in MARKERS:
    col = f"median_error__{mk}"
    rows[mk] = {
        "zero_exactly": float((sh_p[col] == 0).mean()),
        "sampling_med": sh_p[col].median(), "sampling_mean": sh_p[col].mean(),
        "tech_2v3_med": c23[col].median(), "tech_2v3_mean": c23[col].mean(),
        "DMSO1v2_nonPI": c12[c12.stim != "PI"][col].median(),
        "DMSO1v2_PI": c12[c12.stim == "PI"][col].median(),
    }
per_marker = pd.DataFrame(rows).T.sort_values("tech_2v3_mean", ascending=False)
print("=== per-marker floor: |median(A) - median(B)|, pooled scope ===")
print(per_marker.to_string(float_format=fmt))

In [ ]:
# Direction. The floors above are absolute values; the signed offset between two control
# wells is read straight off batchcorrection_2.csv, whose correctedMedian is per
# (patient, treatment, stim, marker, population) -- i.e. the table the runtime correction
# shifts each group onto. Positive d_i_j = control i sits above control j on that marker.
bcm = pd.read_csv(REPO / "cross_validation/batchcorrection_2.csv")
bcm = bcm[bcm.Treatment.isin(["DMSO1", "DMSO2", "DMSO3"]) & bcm.Marker.isin(MARKERS)]
wide = (bcm.pivot_table(index=["Patient", "Stim", "Marker", "Population"],
                        columns="Treatment", values="correctedMedian").dropna())
for a, b in [("DMSO1", "DMSO2"), ("DMSO1", "DMSO3"), ("DMSO2", "DMSO3")]:
    wide[f"d_{a[-1]}_{b[-1]}"] = wide[a] - wide[b]
dcols = ["d_1_2", "d_1_3", "d_2_3"]

pi = wide.xs("PI", level="Stim")
non = wide[wide.index.get_level_values("Stim") != "PI"]
out = pd.concat({"PI": pi.groupby("Marker")[dcols].median(),
                 "other stims": non.groupby("Marker")[dcols].median()}, axis=1)
# How reproducible is that sign across the 19 patients x populations?
out[("PI", "sign_consistency")] = pi.groupby("Marker").apply(
    lambda s: ((s.d_1_2 > 0) == (s.d_1_2.median() > 0)).mean())
print("=== SIGNED offset between control wells (correctedMedian), median over patients x pops ===")
print(out.to_string(float_format=lambda v: f"{v:+8.4f}"))

**Per marker the floor has a direction, and a reproducible one.** Under `PI`, `DMSO1` sits
*above* both other controls on 11 of 13 markers — `pERK` +0.23, `pS6` +0.24, `pMK2` +0.16,
`pCREB` +0.12, `pp38` +0.10 against `DMSO2` — and the sign holds in 85–97% of
(patient × population) groups for the response markers. `HLADR` goes the other way, also
consistently. Under the other stims the same offsets are small and inconsistent.

That is precisely why a median-shift correction works on it: a stable, signed, per-marker offset
is exactly what `correctedMedian` encodes and what `batch_correct` removes.

**`mmd` has no direction, and you cannot do arithmetic with the floor.** Two reasons:

1. It is a distance — non-negative, sign-free. "The plate is +0.008 away" has no meaning.
2. `compute_scalar_mmd` returns `k(x,x).mean() + k(y,y).mean() − 2k(x,y).mean()`, which is the
   biased estimate of **MMD², not MMD**. So these numbers are already in squared-distance units.

The consequence for a number like "raw effect 0.013 against a floor of 0.008": if the drug
perturbation and the plate offset were independent directions in the RKHS, squared distances
would add, so the observed 0.013 ≈ drug + 0.008 plate, and the drug's own contribution is about
**0.005 — you subtract, and the effect shrinks.** You never add. And that is only an
approximation: if the drug moves the cells *along* the same direction as the plate offset the
cross-term is not zero and they can reinforce or partly cancel.

The defensible way to remove a floor is not scalar arithmetic on MMD but applying the signed
per-marker shift to the cells and rescoring — which is what the probe above does (PI,
`DMSO1` v `DMSO2`: 0.0729 → 0.0097).

**One thing this does *not* say.** A drug is compared against **its own** plate control, so a
plate-wide offset affecting drug wells and control wells alike cancels to first order inside that
comparison. The offsets above bite when comparing *across* drugs referenced to different
controls — `LPZ`/`SA` (on `DMSO1`) against `CHT`/`MF`/`PRA`/`MAP` (on `DMSO2`) — which is exactly
the comparison the drug-arm ranking makes.

---

# Part C — the drug arm (2026-09-20)

## 11.0 What ran

`--arm drug` maps **DMSO → drug *within* a stim**, against the paper's **existing** 1512
checkpoints (9 drugs × 168 pairs, `results_drug/cross_validation_drug_OG13`). **Nothing was
retrained** — those maps already exist; only the baselines are fit. Drug-specific loading —
plate-control resolution via `use_DMSO`, runtime batch correction from `batchcorrection_2.csv`,
drug-grouped 80/20 split — lives in `baselines/drug_data.py`.

```bash
# produced simco_work/baselines/pooled_drug/ — 28 CSVs, 18,128 rows, ~9x the stim arm's cost
NPROC=3 bash baselines/run_pooled_drug.sh
LOWMEM=1 bash baselines/run_pooled_drug.sh Granulocytes   # OOMs (rc=137) without --low-memory

W=/scratch/users/bmerwan/simco_work/baselines
python -m baselines.summarize --indir $W/pooled_drug --outdir $W/summary_pooled_drug_cell       --weighting cell
python -m baselines.summarize --indir $W/pooled_drug --outdir $W/summary_pooled_drug_unweighted --weighting unweighted
```

Two traps recorded here because both cost time:

1. **The drug rows carry a `drug` column that stim rows do not** — the two dirs are *not*
   schema-compatible. `summarize.py` adds `drug` to its pairing keys automatically
   (`keys_for()`); a hand-rolled pivot that forgets it collapses nine drugs into one cell.
2. **Granulocytes OOMs** in a 32 GB allocation: the single-pass load peaks >23 GB. `--low-memory`
   reads one stim at a time (~7× the I/O, verified bit-identical), peaking ~19 GB. That is why
   its selection-leak CSV was once *missing* rather than merely unfinished.

## 11.1 The finding is about effect size, not about which method wins

In [ ]:
drug = load_dir(POOLED_DRUG)
dcell = keep_arm(drug, "cell")
dlive, dnull, deffect = pair_split_live_null(dcell)

print(f"{len(drug):,} rows, {drug.groupby(['stim', 'celltype', 'drug']).ngroups} "
      f"(stim, celltype, drug) pairs, {drug.drug.nunique()} drugs\n")
print(f"median identity MMD, drug arm : {deffect.median():.5f}")
print(f"median identity MMD, stim arm : {effect.median():.5f}")
print(f"                        ratio : {effect.median() / deffect.median():.0f}x smaller\n")
n_dlive = int(deffect.ge(0.005).sum())
print(f"live at the same 0.005 threshold: {n_dlive} of {len(deffect)} "
      f"({n_dlive / len(deffect):.1%})   vs the stim arm's {n_live}/168 ({n_live / 168:.0%})")

In [ ]:
print("=== effect size by drug (identity MMD over that drug's 168 pairs) ===")
print(deffect.groupby("drug").agg(["median", "max", "count"]).to_string(float_format=fmt))

livepairs = deffect[deffect >= 0.005]
print("\n=== where the 48 live pairs are ===")
for level in ["stim", "drug", "celltype"]:
    vc = livepairs.index.get_level_values(level).value_counts()
    print(f"by {level:9s}: {dict(vc.head(6))}")
print("\nGranulocytes -- the largest celltype -- max identity MMD over all its drug pairs:",
      f"{deffect.xs('Granulocytes', level='celltype').max():.4f}")

**The drugs barely move these cells.** A 48× smaller perturbation than a stimulation, and only
3.2% of pairs clear the bar that 76% of stim pairs cleared. Every drug's median is below 0.0011.

Two consequences:

- `identity` here means "the drug does nothing" — and over all 1512 pairs it is **a coin flip
  against CellOT** (next cell). That is a statement about the 97% of pairs with no effect to
  predict, not about the model. **No aggregate over all 1512 pairs is worth quoting.**
- The 48 live pairs are extremely concentrated: **46 of 48 are the PI stim**, 39 of 48 are two
  drugs (LPZ 23, SA 16), and **Granulocytes has none at all**.

In [ ]:
print("=== paired win rate vs CellOT, ALL 1512 pairs, cell-weighted -- DO NOT QUOTE ===")
print(paired_win_rates(dcell).to_string(float_format=pct))
print("\n^ identity sits at ~0.50 on four of five metrics and 0.68 on corr_error: with 97% of")
print("  pairs holding no effect, 'the drug does nothing' ties a trained transport map.")

### Reading §11.1

**The stim arm's shape repeats** — CellOT keeps the joint objective (`mmd`: it wins 30/48
outright, and the best baseline, `quantile_map`, takes it in only 29.2% of pairs) and loses the
marker-wise ones. But it loses them to **`quantile_map`** (`w1` 83.3%, `median_error` 77.1%,
`mean_error` 72.9%) rather than to `median_shift`/`gauss_ot`.

**Do not look here for the stim arm's sharpest result.** `data.metric` is **absent from all 1512
drug configs**, so `metric_type` falls back to MMD (`train.py:190-193`): the drug arm selects its
checkpoints on **MMD, the one metric CellOT wins**. There is no "a constant shift beats it on its
own selection criterion" counterpart on this arm.

**The weighting qualifier matters more here than on the stim arm.** Unweighted, `quantile_map`'s
`w1` win rate falls 83.3% → 27.1% and `gauss_ot`'s `corr_error` 72.9% → 0%. With only 48 pairs the
cell-weighted rates rest on a handful of large pairs — **indicative, not the stim arm's 127-pair
statements.**

**And, in fairness to CellOT:** restricted to those 48 live pairs, `model.pt` beats `identity` on
MMD in **94%** of them (§12). Where there *is* a drug effect, CellOT does capture it.

---

## 12. The selection leak — `model.pt` vs `last.pt`

Neither protocol above can remove the one leak that remains: CellOT's shipped `model.pt` is the
**argmin over ~200 evaluations on the same test cells everyone is scored on**
(`train.py:255-265`, every `eval_freq: 250` steps). `last.pt` is the final step, chosen by
nothing. The gap between them bounds how much of CellOT's reported edge is that argmin.

```bash
bash baselines/run_selection_leak.sh              # stim arm  -> selection_leak_stim/
ARM=drug bash baselines/run_selection_leak.sh     # drug arm  -> selection_leak_drug/
python -m baselines.summarize_selection_leak --indir $W/selection_leak_stim --outdir $W/summary_selection_leak_stim
python -m baselines.summarize_selection_leak --indir $W/selection_leak_drug --outdir $W/summary_selection_leak_drug
```

In [ ]:
for name, label in [("summary_selection_leak_stim", "STIM arm, 168 pairs"),
                    ("summary_selection_leak_drug", "DRUG arm, 1512 pairs")]:
    d = WORK / name
    print(f"=== {label} ({name}) ===")
    gap = pd.read_csv(d / "gap_live.csv")
    print(gap.to_string(index=False, float_format=fmt))
    base = gap.set_index("metric")
    mmd_pct = base.loc["mmd", "mean_gap"] / base.loc["mmd", "model_pt"]
    print(f"  mmd gap as a share of model.pt's own error: {mmd_pct:.1%}")
    if (d / "vs_identity.csv").exists():
        print(" ", (d / "vs_identity.csv").read_text().replace("\n", " | ").strip())
    print()

**Stim arm: the leak is small** — the `mmd` gap is 1.2% of `model.pt`'s own error, and
`model.pt` is better than `last.pt` on only 52% of pairs on `mmd`/`w1` (a coin flip). The one
metric where selection clearly bites is `median_error`, 63.8% — which is exactly the metric that
does the selecting on this arm.

**Drug arm: the leak is ~6× bigger** — `mmd` gap 0.000188 on a base of 0.00286 (**6.6%**),
`model.pt` better in 76%. Expected, because on this arm the argmin runs on MMD itself, the metric
being reported.

Neither gap is large enough to overturn §10.3 or §11.1: the margins by which `median_shift` and
`gauss_ot` beat CellOT are far wider than the margin by which `model.pt` beats `last.pt`.

---

## 13. Regenerating the tables above

**Cheap** (seconds) — rebuild all four pooled summary dirs from the per-celltype CSVs. Run this
if anything above disagrees with a file on disk:

In [ ]:
%%bash
cd /scratch/users/bmerwan/simco
W=/scratch/users/bmerwan/simco_work/baselines
set -e
python -m baselines.summarize --indir $W/pooled      --outdir $W/summary_pooled_cell            --weighting cell       | tail -3
python -m baselines.summarize --indir $W/pooled      --outdir $W/summary_pooled_unweighted      --weighting unweighted | tail -3
python -m baselines.summarize --indir $W/pooled_drug --outdir $W/summary_pooled_drug_cell       --weighting cell       | tail -3
python -m baselines.summarize --indir $W/pooled_drug --outdir $W/summary_pooled_drug_unweighted --weighting unweighted | tail -3

**Expensive — do not run in the notebook.** Needs its own allocation; hours.

```bash
ml load python/3.9.0 && source /scratch/users/bmerwan/venv-cellot/bin/activate
cd /scratch/users/bmerwan/simco

NPROC=4 bash baselines/run_pooled.sh                      # stim arm, ~2 h  -> pooled/
bash baselines/run_pooled.sh Granulocytes                 # 30 GB outlier, alone
NPROC=3 bash baselines/run_pooled_drug.sh                 # drug arm, ~9x   -> pooled_drug/
LOWMEM=1 bash baselines/run_pooled_drug.sh Granulocytes   # OOMs without it
bash baselines/run_selection_leak.sh                      # -> selection_leak_stim/
ARM=drug bash baselines/run_selection_leak.sh             # -> selection_leak_drug/
```

All launchers skip a celltype whose CSV already exists, so a walltime kill is resumable.

**Dirs that are not schema-compatible — never `--indir` across them:** `lopo`, `lopo_test`,
`lopo_test_v2` predate `gauss_ot`/`linear_median*`; `pooled` has no `drug` column and
`pooled_drug` does; `run_matched.sh` defaults to `lopo_test_v3` because its resume check only
tests that a CSV exists.